# 04 — Обучение uplift T-learner (treatment + control)

Цель:
- обучить две модели:
  - p(click | treatment = 1)
  - p(click | treatment = 0)
- сохранить их в `models/`,
- получить базовые метрики качества (AUC) для обеих частей T-learner.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_prep.feature_engineering import load_ml_dataset, prepare_features
from src.models.uplift_treatment import (
    train_treatment_model,
    save_treatment_model,
)
from src.models.uplift_control import (
    train_control_model,
    save_control_model,
)
from src.utils.config import (
    UPLIFT_TREATMENT_MODEL_PATH,
    UPLIFT_CONTROL_MODEL_PATH,
)

In [ ]:
df_ml = load_ml_dataset()
df_ml.shape, df_ml.head()

In [ ]:
df_ml["treatment"].value_counts(normalize=True), df_ml["conversion"].value_counts(normalize=True)

In [ ]:
# срез treatment = 1
df_t = df_ml[df_ml["treatment"] == 1].copy()
df_t.shape
X_t, y_t, ids_t, meta_t = prepare_features(df_t)
X_t.shape, y_t.mean()
# train/valid split
X_tr_t, X_val_t, y_tr_t, y_val_t = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t
)

# catboost-параметры те же, что в DEFAULT_TREATMENT_PARAMS
treatment_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3.0,
    "iterations": 500,
    "random_seed": 42,
    "verbose": 100,
}

cat_features_t = [
    col for col in meta_t["categorical_features"]
    if col in X_tr_t.columns
]

model_t = CatBoostClassifier(**treatment_params)
model_t.fit(
    X_tr_t,
    y_tr_t,
    eval_set=(X_val_t, y_val_t),
    cat_features=cat_features_t,
    use_best_model=True,
)
# AUC на валидации
p_val_t = model_t.predict_proba(X_val_t)[:, 1]
auc_t = roc_auc_score(y_val_t, p_val_t)
auc_t

In [ ]:
from src.models.uplift_treatment import save_treatment_model

save_treatment_model(model_t, meta_t)
UPLIFT_TREATMENT_MODEL_PATH

In [ ]:
df_c = df_ml[df_ml["treatment"] == 0].copy()
df_c.shape
X_c, y_c, ids_c, meta_c = prepare_features(df_c)
X_c.shape, y_c.mean()
X_tr_c, X_val_c, y_tr_c, y_val_c = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

control_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "depth": 6,
    "learning_rate": 0.05,
    "l2_leaf_reg": 3.0,
    "iterations": 500,
    "random_seed": 43,
    "verbose": 100,
}

cat_features_c = [
    col for col in meta_c["categorical_features"]
    if col in X_tr_c.columns
]

model_c = CatBoostClassifier(**control_params)
model_c.fit(
    X_tr_c,
    y_tr_c,
    eval_set=(X_val_c, y_val_c),
    cat_features=cat_features_c,
    use_best_model=True,
)
p_val_c = model_c.predict_proba(X_val_c)[:, 1]
auc_c = roc_auc_score(y_val_c, p_val_c)
auc_c

In [ ]:
from src.models.uplift_control import save_control_model

save_control_model(model_c, meta_c)
UPLIFT_CONTROL_MODEL_PATH

In [ ]:
from src.models.uplift_treatment import load_treatment_model, predict_treatment_proba
from src.models.uplift_control import load_control_model, predict_control_proba

model_t_loaded, meta_t_loaded = load_treatment_model()
model_c_loaded, meta_c_loaded = load_control_model()

# маленький sample из общего df_ml
df_sample = df_ml.sample(5, random_state=7).reset_index(drop=True)
X_sample, y_sample, ids_sample, meta_sample = prepare_features(df_sample)

p_treat_sample = predict_treatment_proba(model_t_loaded, X_sample, meta_t_loaded)
p_control_sample = predict_control_proba(model_c_loaded, X_sample, meta_c_loaded)

pd.DataFrame({
    "client_id": ids_sample["client_id"],
    "offer_id": ids_sample["offer_id"],
    "p_treat": p_treat_sample,
    "p_control": p_control_sample,
    "uplift_raw": p_treat_sample - p_control_sample,
})

## Итоги обучения uplift-моделей

1. Обучены две модели:
   - treatment-модель: p(click | treatment=1), AUC ≈ `auc_t`;
   - control-модель: p(click | treatment=0), AUC ≈ `auc_c`.
2. Модели и мета-информация о фичах сохранены в:
   - `models/uplift_treatment.cbm` + `uplift_treatment_meta.json`
   - `models/uplift_control.cbm` + `uplift_control_meta.json`
3. Быстрый sanity-check на сэмпле показывает адекватные вероятности
   и ненулевой uplift = p_treat – p_control.
4. На следующем шаге:
   - реализуем модуль `uplift_scoring.py` (подсчёт uplift и expected_gain),
   - сравним uplift-стратегию с rule-based baseline (ноутбук `05_uplift_vs_rule_based.ipynb`),
   - затем обернём всё в API.